# Anomaly Detection and Operational Risk Analysis

Identifying unusual city/time conditions associated with elevated rider-experience risk.

In [ ]:
import pandas as pd
import plotly.express as px
from pathlib import Path

from roadies.ingestion.loaders import load_csv
from roadies.features.demand_supply import engineer_demand_supply_features
from roadies.features.surge import engineer_surge_features
from roadies.features.acceptance import engineer_acceptance_features
from roadies.features.cancellation import engineer_cancellation_features
from roadies.features.experience import engineer_experience_features
from roadies.features.demand_period import classify_high_demand
from roadies.analysis.anomaly import (
    detect_anomalies,
    detect_city_relative_anomalies,
    classify_risk,
    identify_risk_periods,
    count_city_anomalies,
)

In [ ]:
# Load and engineer features
df = load_csv(Path('data/raw/rides.csv'))
df, _ = engineer_demand_supply_features(df)
df, _ = engineer_surge_features(df)
df, _ = engineer_acceptance_features(df)
df, _ = engineer_cancellation_features(df)
df, _ = engineer_experience_features(df)
df, _ = classify_high_demand(df)
print(f'Dataset: {len(df)} rows')

In [ ]:
# Global anomalies
anomalies = detect_anomalies(df)
print(f'Global anomalies: {len(anomalies)}')
for a in anomalies[:5]:
    print(f'  {a.metric}: {a.value:.2f} (baseline={a.baseline:.2f}, severity={a.severity})')

In [ ]:
# Risk classification
risk_df = classify_risk(df)
print('Risk distribution:')
print(risk_df['risk_level'].value_counts())

In [ ]:
# City anomaly frequency
city_anomalies = count_city_anomalies(df)
print('City anomaly rates:')
print(city_anomalies[['city', 'anomaly_rate', 'critical_count']].to_string())

In [ ]:
# Risk periods
periods = identify_risk_periods(df)
high_risk = periods[periods['is_high_risk'] == True]
print(f'High-risk periods: {len(high_risk)}')